# Random Forests

Last time we saw that single decision trees have a fundamental problem: shallow trees underfit and deep trees overfit. No single depth works for all parts of the feature space. We ended with a question - what if we grew many trees and averaged their predictions?

That's exactly what a random forest does. The idea is simple: train many trees, each on a slightly different version of the data, and let them vote. Individual trees still overfit, but they overfit *differently*. Averaging cancels the noise while preserving the signal. The result is a model that's dramatically more stable than any single tree.

This lecture covers how random forests work, why they work, and how to use them effectively. We'll stay on the California Housing dataset from last time so you can directly compare RF performance with the single trees you already know.

Changes after Day 22 lecture (2026-03-31):

- [x] Clarify that MDI uses MSE reduction (not Gini) for regression trees
- [x] Add p/3 (regression) and sqrt(p) (classification) recommendation for `max_features` from ISL/ESL
- [x] Add note distinguishing bootstrap sampling in RF from over/under-sampling for imbalanced data

## Setup

In [ ]:
%matplotlib inline

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.datasets import fetch_california_housing, make_moons
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import BaggingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.inspection import DecisionBoundaryDisplay, permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

In [ ]:
# Same dataset and split as 12b
california = fetch_california_housing()
X, y = california.data, california.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

print(f"Training: {X_train.shape[0]:,} samples, {X_train.shape[1]} features")
print(f"Test:     {X_test.shape[0]:,} samples")

---

## Part 1: Many Trees Are Better Than One

### Why combining weak learners works

Before touching sklearn, let's build the intuition with a simple experiment. Imagine you have a coin that lands heads 51% of the time - barely better than fair. One flip tells you almost nothing. But flip it 1,000 times and take the majority outcome: the probability that heads wins the majority is much higher than 51%.

In [ ]:
# Simulate: how does majority-vote accuracy scale with number of "voters"?
rng = np.random.default_rng(42)
n_voters_range = np.arange(1, 1001, 2)  # odd numbers to avoid ties
p_correct = 0.51  # each voter is barely better than chance

majority_acc = []
for n in n_voters_range:
    # Simulate 10,000 elections with n voters
    votes = rng.binomial(1, p_correct, size=(10_000, n))
    majority = votes.mean(axis=1) > 0.5
    majority_acc.append(majority.mean())

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(n_voters_range, majority_acc)
ax.axhline(y=0.51, color="gray", linestyle="--", alpha=0.5, label="Single voter (51%)")
ax.set_xlabel("Number of voters")
ax.set_ylabel("Majority-vote accuracy")
ax.set_title("Weak Learners + Majority Vote")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"  1 voter:    {majority_acc[0]:.1%}")
print(f" 101 voters:  {majority_acc[50]:.1%}")
print(f"1001 voters:  {majority_acc[-1]:.1%}")

With enough voters, the majority is almost always right - even though each individual voter is barely better than a coin flip. This is the law of large numbers at work, and it's the principle behind ensemble methods. Each decision tree in a random forest is a "weak learner" - individually noisy and overfit, but collectively powerful. The key requirement is that their errors are *somewhat independent*. If every voter makes the same mistake, more voters don't help.

### The single-tree baseline

In 12b we saw that even a tuned decision tree has a gap between training and test performance. Let's establish the baseline we're trying to beat.

In [ ]:
# Single tree - same as 12b Part 3
dt = DecisionTreeRegressor(max_depth=10, random_state=42)
dt.fit(X_train, y_train)

dt_train_rmse = np.sqrt(mean_squared_error(y_train, dt.predict(X_train)))
dt_test_rmse = np.sqrt(mean_squared_error(y_test, dt.predict(X_test)))

print(f"Single tree (depth=10):")
print(f"  Train RMSE: {dt_train_rmse:.4f}")
print(f"  Test RMSE:  {dt_test_rmse:.4f}")
print(f"  Gap:        {dt_test_rmse - dt_train_rmse:.4f}")

That gap is overfitting. The tree memorizes patterns in the training data that don't generalize. We could tune `max_depth` or `min_samples_leaf` to reduce it, but we're always trading off: simpler trees have less overfitting but also miss real structure.

### The random forest fix

A random forest takes a different approach. Instead of building one careful tree, it builds many careless ones and averages them.

Each tree in the forest is trained on a *bootstrap sample* - a random draw of N observations from the training data, sampled with replacement. With replacement means some observations appear multiple times while others are left out entirely. On average, each bootstrap sample contains about 63% of the unique training observations. The other 37% are called *out-of-bag* (OOB) samples - we'll come back to those shortly.

The 63%/37% split comes from probability theory: the chance of any single observation being selected at least once in N draws with replacement is $1 - (1 - 1/N)^N$, which approaches $1 - 1/e \approx 0.632$ as N grows. That leftover 37% is what makes OOB estimation possible.

Because each tree sees a different subset of the data, they learn slightly different patterns and make different errors. When we average their predictions, the individual errors tend to cancel out while the genuine signal reinforces. This is the same principle behind polling: any one poll is noisy, but averaging many polls gives a better estimate.

In [ ]:
# Random Forest with 100 trees
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

rf_train_rmse = np.sqrt(mean_squared_error(y_train, rf.predict(X_train)))
rf_test_rmse = np.sqrt(mean_squared_error(y_test, rf.predict(X_test)))

print(f"Random Forest (100 trees):")
print(f"  Train RMSE: {rf_train_rmse:.4f}")
print(f"  Test RMSE:  {rf_test_rmse:.4f}")
print(f"  Gap:        {rf_test_rmse - rf_train_rmse:.4f}")
print()
print(f"Improvement over single tree: {dt_test_rmse - rf_test_rmse:.4f} Test RMSE")

The test error drops substantially - that's the number that matters for generalization. The train RMSE is even lower because each tree in the forest still memorizes its own bootstrap sample, but the *test* predictions benefit from averaging many different memorizers. The improvement over the single tree is real: the forest finds more robust patterns by combining diverse perspectives on the data.

### Seeing the averaging effect

The step-function plot from 12b showed how a single tree approximates a continuous relationship with flat segments. Let's plot predictions from several individual trees in the forest against the ensemble average. We'll use `MedInc` (the most important feature from 12b) as the x-axis.

In [ ]:
# Extract individual tree predictions along MedInc
feat_idx = list(california.feature_names).index("MedInc")
X_range = np.zeros((200, X.shape[1]))
X_range[:, feat_idx] = np.linspace(X[:, feat_idx].min(), X[:, feat_idx].max(), 200)
# Set other features to their training-set medians
for j in range(X.shape[1]):
    if j != feat_idx:
        X_range[:, j] = np.median(X_train[:, j])

fig, ax = plt.subplots(figsize=(10, 5))

# Plot individual trees (first 6)
for i, tree in enumerate(rf.estimators_[:6]):
    y_tree = tree.predict(X_range)
    ax.plot(X_range[:, feat_idx], y_tree, alpha=0.25, linewidth=1, color="steelblue")

# Plot ensemble average
y_rf = rf.predict(X_range)
ax.plot(X_range[:, feat_idx], y_rf, color="tomato", linewidth=2.5, label="Forest average")
ax.plot([], [], color="steelblue", alpha=0.4, label="Individual trees (6 shown)")

ax.set_xlabel("MedInc")
ax.set_ylabel("Predicted House Value (\\$100k)")
ax.set_title("Individual Trees vs. Forest Average")
ax.legend()
plt.tight_layout()
plt.show()

Each blue line is a single tree's prediction - jagged step functions that disagree on details. The red line is the forest's average - smoother, more stable, and closer to the true underlying pattern. This is variance reduction in action: the trees' individual quirks cancel out, while the trend they share is reinforced.

### A note on bootstrap resampling

You've actually used bootstrap resampling before. In the debugging workshop (07b), you refit OLS and Ridge on 200 bootstrap resamples to measure coefficient instability under collinearity. There, bootstrap measured *how much an estimate changes across different samples of the data*.

Bootstrap resampling is a general-purpose statistical technique with applications well beyond random forests. In statistics, it's used for confidence intervals, hypothesis testing, and estimating the variability of any statistic when a formula doesn't exist. The core idea is always the same: resample your data with replacement to simulate having multiple datasets, or to estimate how results would hold up if you had more data from the same distribution.

Don't confuse bootstrap sampling with the over/under-sampling techniques from the imbalanced data lecture (11b). They serve different goals. Bootstrap sampling in RF preserves the original class distribution - each bootstrap sample has roughly the same balance as the training data. The purpose is to create *diversity* among trees so their errors are independent. Over/under-sampling deliberately *changes* the class distribution to help the model learn from rare events. Different problems, different tools.

### Why random *features* matter

Bootstrap sampling alone helps, but there's a ceiling. If one feature is much stronger than the rest, every tree will split on it first regardless of the bootstrap sample. The trees end up correlated - they make similar predictions and similar errors - and averaging correlated estimates doesn't reduce variance as much as averaging independent ones.

Random forests fix this by adding a second source of randomness: at each split, the tree considers only a random subset of features. This forces some trees to find alternative splitting strategies, which decorrelates their predictions. Individual trees may get slightly worse, but the ensemble improves because their errors are more independent.

For regression, the default is to consider all features at each split (`max_features=1.0`). For classification, the default is $\sqrt{p}$ features. You can control this directly - lower values mean more diversity between trees.

In [ ]:
# RF with all features (default for regression) vs sqrt features per split
rf_sqrt = RandomForestRegressor(
    n_estimators=100, max_features="sqrt", random_state=42
)
rf_sqrt.fit(X_train, y_train)
rf_sqrt_test_rmse = np.sqrt(mean_squared_error(y_test, rf_sqrt.predict(X_test)))

print(f"Single tree:                    {dt_test_rmse:.4f}")
print(f"RF (default, all features):     {rf_test_rmse:.4f}")
print(f"RF (sqrt features per split):   {rf_sqrt_test_rmse:.4f}")

We can see the decorrelation effect directly by looking at which feature each tree chooses for its *first* split (the root node).

In [ ]:
# Which feature does each tree split on first?
def root_features(forest, feature_names):
    """Count which feature is used at the root node of each tree."""
    counts = {}
    for tree in forest.estimators_:
        feat = feature_names[tree.tree_.feature[0]]
        counts[feat] = counts.get(feat, 0) + 1
    return counts

# RF with all features (default for regression) vs sqrt
rf_all = RandomForestRegressor(n_estimators=200, max_features=1.0, random_state=42)
rf_all.fit(X_train, y_train)
roots_all = root_features(rf_all, california.feature_names)

rf_div = RandomForestRegressor(n_estimators=200, max_features="sqrt", random_state=42)
rf_div.fit(X_train, y_train)
roots_div = root_features(rf_div, california.feature_names)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

feats = california.feature_names
ax1.barh(feats, [roots_all.get(f, 0) for f in feats])
ax1.set_xlabel("Trees using this feature at root")
ax1.set_title("max_features=1.0 (all features)")

ax2.barh(feats, [roots_div.get(f, 0) for f in feats])
ax2.set_xlabel("Trees using this feature at root")
ax2.set_title('max_features="sqrt" (~3 features)')

plt.tight_layout()
plt.show()

With all features available, nearly every tree splits on `MedInc` first - they're all seeing the same dominant signal and making correlated predictions. With `sqrt` features, some trees are forced to start with `AveOccup`, `Latitude`, or other features, producing genuinely different tree structures. This diversity is what makes the ensemble stronger.

Now look at the RMSE comparison above. RF with default settings (`max_features=1.0`) considers all features at every split, so the random feature subset mechanism is effectively turned off. Setting `max_features="sqrt"` activates decorrelation and improves the result. On California Housing, where `MedInc` dominates, forcing trees to sometimes split without it creates genuine diversity.

Why does `RandomForestRegressor` default to all features while `RandomForestClassifier` defaults to `sqrt`? Regression targets have continuous structure that benefits from access to every feature at each split. Classification boundaries are often driven by a few key features, so restricting the available set costs less and the decorrelation benefit is larger. In practice, `max_features` is worth tuning on any dataset - it's the most important RF hyperparameter.

### Visualizing the effect: DT vs RF on classification

Everything so far has been regression on California Housing. Let's switch to a classification dataset where we can *see* what averaging does to decision boundaries. `make_moons` generates two interleaving crescents with noise - a non-linear boundary that trees approximate with rectangles.

In [ ]:
X_moons, y_moons = make_moons(n_samples=300, noise=0.25, random_state=42)

dt_moons = DecisionTreeClassifier(max_depth=7, random_state=42)
rf_moons = RandomForestClassifier(n_estimators=300, random_state=42)

dt_moons.fit(X_moons, y_moons)
rf_moons.fit(X_moons, y_moons)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

for ax, model, title in [
    (ax1, dt_moons, f"Single Tree (depth={dt_moons.get_depth()})"),
    (ax2, rf_moons, "Random Forest (300 trees)"),
]:
    DecisionBoundaryDisplay.from_estimator(
        model, X_moons, ax=ax, cmap="RdYlBu", alpha=0.3, response_method="predict",
    )
    ax.scatter(
        X_moons[:, 0], X_moons[:, 1], c=y_moons,
        cmap="RdYlBu", edgecolor="k", s=20,
    )
    ax.set_xlim(-2, 3)
    ax.set_ylim(-1.25, 2)
    ax.set_title(title)

plt.tight_layout()
plt.show()

The single tree (left) produces jagged, rectangular boundaries that follow the noise in the training data - you can see tiny isolated regions where the tree carved out exceptions for individual points. The random forest (right) produces a much smoother boundary that follows the true crescent shape. Both use the same `DecisionBoundaryDisplay` tool you saw in the SVM lecture (Day 20) and the tree lecture (12b).

This is the same variance-reduction story as the regression plot above, but in a form you can see directly: averaging many jagged boundaries produces a smooth one.

The `RandomForestClassifier` API is identical to `RandomForestRegressor` - swap the class name and everything else works the same way. For classification, `max_features` defaults to `"sqrt"` rather than `1.0`, which means the decorrelation mechanism is on by default.

---

## Part 2: Practical RF

### Out-of-bag estimation

Each tree in the forest leaves out about 37% of the training data (the OOB samples). We can use those left-out samples to estimate generalization error without a separate validation set or cross-validation.

For each training observation, collect the predictions from only the trees that *didn't* include it in their bootstrap sample. Average those predictions and compare with the true value. The result is an honest performance estimate - each prediction comes from trees that never saw that observation during training.

Here's how it works for a small example with 5 trees and 6 observations. Each tree trains on a bootstrap sample (marked with x) and skips the rest (marked OOB). The OOB prediction for each observation averages only the trees that didn't train on it:

In [ ]:
# Illustrate OOB concept with a small example
fig, ax = plt.subplots(figsize=(8, 3.5))

# Simulated bootstrap membership (rows=observations, cols=trees)
np.random.seed(0)
n_obs, n_trees_demo = 6, 5
membership = np.random.choice([0, 1], size=(n_obs, n_trees_demo), p=[0.37, 0.63])

for i in range(n_obs):
    for j in range(n_trees_demo):
        color = "steelblue" if membership[i, j] else "lightyellow"
        edge = "steelblue" if membership[i, j] else "tomato"
        text = "x" if membership[i, j] else "OOB"
        ax.add_patch(plt.Rectangle((j, n_obs - 1 - i), 0.9, 0.8,
                     facecolor=color, edgecolor=edge, alpha=0.6))
        ax.text(j + 0.45, n_obs - 1 - i + 0.4, text,
                ha="center", va="center", fontsize=9,
                color="white" if membership[i, j] else "tomato",
                fontweight="bold")

ax.set_xlim(-0.3, n_trees_demo + 0.2)
ax.set_ylim(-0.3, n_obs + 0.2)
ax.set_xticks([j + 0.45 for j in range(n_trees_demo)])
ax.set_xticklabels([f"Tree {j+1}" for j in range(n_trees_demo)])
ax.set_yticks([n_obs - 1 - i + 0.4 for i in range(n_obs)])
ax.set_yticklabels([f"Obs {i+1}" for i in range(n_obs)])
ax.set_title("Bootstrap membership: x = trained on, OOB = left out")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

For Obs 1, only the trees marked OOB contribute to its OOB prediction. Because those trees never trained on Obs 1, the prediction is honest - just like a held-out test set, but computed automatically as a byproduct of training.

In [ ]:
rf_oob = RandomForestRegressor(
    n_estimators=200, oob_score=True, random_state=42
)
rf_oob.fit(X_train, y_train)

# Compute OOB RMSE from the OOB predictions for direct comparison
oob_rmse = np.sqrt(mean_squared_error(y_train, rf_oob.oob_prediction_))

# Compare OOB with CV and test set - all in RMSE
cv_scores = cross_val_score(
    RandomForestRegressor(n_estimators=200, random_state=42),
    X_train, y_train, cv=5, scoring="neg_root_mean_squared_error",
)

print(f"OOB RMSE:   {oob_rmse:.4f}")
print(f"CV RMSE:    {-cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")
print(f"Test RMSE:  {np.sqrt(mean_squared_error(y_test, rf_oob.predict(X_test))):.4f}")

All three estimates are in the same ballpark. OOB is faster because the left-out predictions are a byproduct of training, not additional computation. When you need a quick sanity check during exploration, `oob_score=True` is convenient. For final model selection or reporting, CV or a held-out test set is more standard.

### How many trees?

More trees always helps (or at least doesn't hurt) because each new tree contributes an independent estimate. Unlike boosting (Thursday's topic), RF cannot overfit by adding more trees. But there are diminishing returns - and each tree costs compute time.

In [ ]:
n_trees_range = [25, 50, 100, 200, 500]
oob_scores = []
test_rmses = []

for n in n_trees_range:
    model = RandomForestRegressor(
        n_estimators=n, oob_score=True, random_state=42
    )
    model.fit(X_train, y_train)
    oob_scores.append(model.oob_score_)  # R² on out-of-bag samples
    test_rmses.append(
        np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    )

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(n_trees_range, oob_scores, "o-")
ax1.set_xlabel("Number of trees")
ax1.set_ylabel("OOB R²")
ax1.set_title("OOB Score vs. n_estimators")
ax1.grid(True, alpha=0.3)

ax2.plot(n_trees_range, test_rmses, "s-", color="tomato")
ax2.set_xlabel("Number of trees")
ax2.set_ylabel("Test RMSE")
ax2.set_title("Test RMSE vs. n_estimators")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

Performance improves quickly up to about 100 trees, then levels off. Going from 100 to 500 buys almost nothing. In practice, 100-200 trees is a reasonable default. More than that is rarely worth the compute cost unless you're doing a final production fit.

### Other hyperparameters

Beyond `n_estimators` and `max_features`, the most useful RF hyperparameters are:

- `max_depth` - less critical than with single trees. Deep RF trees typically outperform shallow ones because averaging smooths out the overfitting that plagued individual deep trees. The default (no limit) is usually fine.
- `min_samples_leaf` - sets a minimum leaf size. Values of 1-5 are typical. Higher values act as regularization, but RF already regularizes through averaging.
- `n_jobs=-1` - train trees in parallel on all available CPU cores. Trees are independent of each other, so this is pure parallelism with no algorithmic cost. A major practical advantage of RF over sequential methods like boosting.

If you tune one hyperparameter for RF, make it `max_features`. It controls the diversity-strength tradeoff that makes RF work. A common starting point from the statistical learning literature (ISL/ESL): try $p/3$ for regression and $\sqrt{p}$ for classification, where $p$ is the number of features. sklearn's classification default follows this; the regression default (all features) is more conservative and worth overriding.

---

## Part 3: Feature Importance

### MDI: mean decrease in impurity

You've already seen `feature_importances_` on single trees in 12b. For a random forest, it's the same metric averaged across all trees. Each time a feature is used for a split, the reduction in the splitting criterion is accumulated - that's MSE reduction for regression trees, Gini impurity reduction for classification trees. Since we're using `RandomForestRegressor` here, the importance scores reflect how much each feature reduces MSE across the forest. Averaging over 200 trees gives a much more stable estimate than a single tree.

In [ ]:
importance_mdi = rf_oob.feature_importances_
sorted_idx = np.argsort(importance_mdi)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(
    np.array(california.feature_names)[sorted_idx],
    importance_mdi[sorted_idx],
)
ax.set_xlabel("Importance (MDI)")
ax.set_title("Random Forest Feature Importance (MDI)")
plt.tight_layout()
plt.show()

`MedInc` still dominates, as it did with the single tree in 12b. But the estimates are smoother - the forest has seen many different bootstrap samples and feature subsets, so each feature's contribution is measured from multiple angles.

MDI has a known limitation: it's biased toward features with many unique values (high cardinality) and features that are correlated with other informative features. On California Housing this may not be an issue, but on other datasets it can overstate the importance of noisy features.

### Permutation importance: a more reliable alternative

Permutation importance takes a different approach. After the model is trained, shuffle one feature's values in the test set and measure how much worse the predictions get. If performance drops a lot, that feature matters. If it barely changes, the feature isn't contributing much.

This is model-agnostic - it works with any model, not just trees. And it measures importance on actual predictions, not training-time impurity calculations.

Here's what "shuffle" means concretely. Take the first 5 test observations and look at `MedInc`:

In [ ]:
# Show what shuffling a feature looks like
sample = pd.DataFrame(
    X_test[:5], columns=california.feature_names
)[["MedInc", "HouseAge", "AveRooms"]].round(2)

# Shuffle MedInc - randomly reorder its values, breaking the relationship with y
shuffled = sample.copy()
rng_perm = np.random.default_rng(0)
shuffled["MedInc"] = rng_perm.permutation(shuffled["MedInc"].values)

print("Original:")
print(sample.to_string(index=False))
print("\nAfter shuffling MedInc:")
print(shuffled.to_string(index=False))

`MedInc` values are the same set of numbers, just reordered. Each row now has someone else's income paired with its own age and rooms. If the model relied on `MedInc`, its predictions for these rows will change - and get worse. If it didn't rely on `MedInc`, shuffling has no effect. The size of the performance drop measures importance.

The philosophy is deliberately pragmatic: break the relationship between one feature and the target, then measure the damage. A mathematician might object to corrupting data on principle, but the approach is effective precisely because it tests what the model actually uses rather than what the training algorithm computed internally. It's one of the more useful ideas to come out of the ML practitioner community.

In [ ]:
perm_result = permutation_importance(
    rf_oob, X_test, y_test, n_repeats=10, random_state=42
)

sorted_idx_perm = perm_result.importances_mean.argsort()

fig, ax = plt.subplots(figsize=(8, 5))
ax.boxplot(
    perm_result.importances[sorted_idx_perm].T,
    vert=False,
    tick_labels=np.array(california.feature_names)[sorted_idx_perm],
)
ax.set_xlabel("Decrease in R² when feature is shuffled")
ax.set_title("Random Forest Feature Importance (Permutation)")
plt.tight_layout()
plt.show()

The box plot shows variability across the 10 shuffling repeats. Features with narrow boxes and large means are reliably important. Features near zero with wide boxes are noise.

Compare the two rankings - the top features should agree, but the ordering of less important features may differ. Permutation importance is the safer default for anything you'll report or present. MDI is fast and convenient for quick exploration. For your project, use permutation importance.

### Side-by-side comparison

In [ ]:
# Sort both charts by permutation importance (the more trustworthy measure)
shared_order = perm_result.importances_mean.argsort()
feat_names = np.array(california.feature_names)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# MDI - same feature order as permutation
ax1.barh(feat_names[shared_order], importance_mdi[shared_order])
ax1.set_xlabel("Importance (MDI)")
ax1.set_title("Mean Decrease in Impurity")

# Permutation
ax2.barh(feat_names[shared_order], perm_result.importances_mean[shared_order])
ax2.set_xlabel("Decrease in R²")
ax2.set_title("Permutation Importance")

plt.tight_layout()
plt.show()

Both rankings agree on the top features (`MedInc`, `AveOccup`, geographic coordinates) but may disagree on the less important ones. When they diverge, trust permutation importance - it measures actual impact on predictions rather than a training-time surrogate. MDI can inflate the importance of features that happen to produce many splits (high cardinality) or that are correlated with truly important features. Permutation importance is immune to both issues because it tests on held-out data.

---

## Part 4: Baseline Hierarchy

We've been building a progression of models across this course. Each gap in the hierarchy tells a story about the data - not just "which model is best" but *why*.

In [ ]:
models = {
    "Dummy (mean)": DummyRegressor(),
    "Ridge": Ridge(),
    "KNN (scaled)": Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor())]),
    "Decision Tree": DecisionTreeRegressor(max_depth=10, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42),
}

rows = []
for name, model in models.items():
    start = time.time()
    cv = cross_val_score(
        model, X_train, y_train, cv=5,
        scoring="neg_root_mean_squared_error",
    )
    elapsed = time.time() - start
    rows.append({
        "Model": name,
        "CV RMSE (mean)": f"{-cv.mean():.4f}",
        "CV RMSE (std)": f"{cv.std():.4f}",
        "Time (s)": f"{elapsed:.1f}",
    })

hierarchy = pd.DataFrame(rows)
print(hierarchy.to_string(index=False))

In [ ]:
# Visualize the hierarchy - the gaps tell the story
rmse_vals = [float(r["CV RMSE (mean)"]) for r in rows]
model_names = [r["Model"] for r in rows]

fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(model_names, rmse_vals, color="steelblue", edgecolor="black")
ax.set_xlabel("CV RMSE (lower is better)")
ax.set_title("Baseline Hierarchy — California Housing")
ax.invert_yaxis()

# Annotate each bar with its value
for bar, val in zip(bars, rmse_vals):
    ax.text(val + 0.01, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=10)

plt.tight_layout()
plt.show()

Reading the gaps:

| Gap | What it reveals |
|---|---|
| Dummy → Ridge | Linear signal exists in the data |
| Ridge → KNN | Nonlinearity matters - flexible local boundaries help |
| KNN ≈ Decision Tree | Comparable on this dataset - both capture non-linear structure, via different mechanisms |
| Decision Tree → Random Forest | Large improvement - variance reduction through averaging |

KNN and DT land close together here. KNN uses local distance-based averaging; DT uses recursive splitting. They're different strategies for the same problem (non-linearity) and neither dominates the other on every dataset. The real story is the DT → RF jump: averaging 200 trees recovers signal that any single tree misses due to overfitting.

Note that KNN is wrapped in a `Pipeline` with `StandardScaler` - without scaling, KNN performs terribly because California Housing features have very different ranges (income in tens of thousands, latitude around 35). Recall the "when to scale" framework from the Pipeline lecture: distance-based methods need scaling, trees don't.

The DT → RF gap on this dataset is substantial. That's typical - single trees overfit, and averaging fixes it. If this gap were small, your data would be simple enough that ensembles are overkill.

Notice the time column. RF is slower than a single tree because it trains 200 trees. But each tree is independent, so with `n_jobs=-1` the wall-clock time drops proportionally to available cores. This structural parallelism is a practical advantage we'll contrast with boosting, which builds trees sequentially.

## Part 5: The Bigger Picture - Ensemble Methods

What we've been doing - training many models and combining their predictions - is a general strategy called *ensemble learning*. Random forests are one specific ensemble method. There are several others, each with a different approach to the same core idea.

*Bagging* (bootstrap aggregating) is the strategy behind random forests. Train many copies of the same model on different bootstrap samples and average the results. The goal is to reduce variance. `BaggingRegressor` in sklearn implements this for any base estimator - RF is a specialized, optimized version for trees. A related variant called *pasting* samples *without* replacement instead; bagging is more common because the overlap between bootstrap samples is what enables OOB estimation.

The power of `BaggingRegressor` is that it works with *any* model, not just trees. Does bagging always help? It depends on how unstable the base model is. Let's test two extremes: Ridge (low variance, stable) and KNN with k=1 (high variance, memorizes training data - recall from Unit 2 that k=1 is the most overfit setting).

In [ ]:
# Bagged Ridge: 100 copies of a stable model
bag_ridge = BaggingRegressor(
    estimator=Pipeline([("scaler", StandardScaler()), ("ridge", Ridge())]),
    n_estimators=100, random_state=42,
)
bag_ridge.fit(X_train, y_train)

# Bagged KNN k=1: 100 copies of an unstable model
bag_knn = BaggingRegressor(
    estimator=Pipeline([("scaler", StandardScaler()), ("knn", KNeighborsRegressor(n_neighbors=1))]),
    n_estimators=100, random_state=42,
)
bag_knn.fit(X_train, y_train)

# Compare each with its unbagged version
ridge_cv = -cross_val_score(
    Pipeline([("s", StandardScaler()), ("r", Ridge())]),
    X_train, y_train, cv=5, scoring="neg_root_mean_squared_error",
).mean()
knn1_cv = -cross_val_score(
    Pipeline([("s", StandardScaler()), ("k", KNeighborsRegressor(n_neighbors=1))]),
    X_train, y_train, cv=5, scoring="neg_root_mean_squared_error",
).mean()

print(f"{'Model':<20s} {'Single CV RMSE':>15s} {'Bagged Test RMSE':>17s}")
print(f"{'Ridge':<20s} {ridge_cv:>15.4f} {np.sqrt(mean_squared_error(y_test, bag_ridge.predict(X_test))):>17.4f}")
print(f"{'KNN (k=1)':<20s} {knn1_cv:>15.4f} {np.sqrt(mean_squared_error(y_test, bag_knn.predict(X_test))):>17.4f}")

Bagging Ridge does essentially nothing - Ridge is already stable, so averaging 100 copies has nothing to smooth out. Bagging KNN k=1 makes a real difference: each k=1 model memorizes its bootstrap sample differently, and averaging their predictions recovers a much smoother, more generalizable result. This is the same mechanism that makes random forests work - bagging is most powerful with high-variance base learners where individual models disagree substantially.

*Boosting* takes the opposite approach. Instead of training independent models in parallel, it builds models *sequentially* - each new model focuses specifically on the errors the previous ones made. Where bagging reduces variance, boosting reduces bias. The key controls are the number of sequential steps and the learning rate (how aggressively each step corrects). `GradientBoostingRegressor` and XGBoost are the major implementations. We'll cover these on Thursday.

*Stacking* combines different model *types* rather than many copies of the same model. Train a logistic regression, a random forest, and an SVM, then feed their predictions into a meta-model that learns how to combine them. The idea is that diverse models capture different patterns in the data. We'll see `VotingClassifier` and `StackingClassifier` later in the unit.

| Strategy | How it works | Reduces | Key tool |
|---|---|---|---|
| Bagging | Average many copies, trained on bootstrap samples | Variance | `RandomForestRegressor` |
| Boosting | Build sequentially, each model corrects the last | Bias | `GradientBoostingRegressor`, XGBoost |
| Stacking | Combine different model types via a meta-learner | Both (if models are diverse) | `StackingClassifier` |

The baseline hierarchy we built in Part 4 will grow as we add boosted and stacked models over the next two lectures. By the end of the unit, you'll have the full toolkit for your project.

## Summary

| Property | Single Tree | Random Forest |
|---|---|---|
| Overfitting | High - unconstrained trees memorize | Low - averaging cancels noise |
| Scaling required | No | No |
| Parallelizable | N/A | Yes (`n_jobs=-1`) |
| Interpretability | High - trace root to leaf | Low - hundreds of trees |
| Key hyperparameter | `max_depth` | `max_features`, `n_estimators` |

New APIs:

| API | Purpose |
|---|---|
| `RandomForestRegressor` | Ensemble of trees with bootstrap + random features |
| `BaggingRegressor` | General-purpose bootstrap aggregation |
| `oob_score` / `oob_score_` | Free validation from bootstrap |
| `permutation_importance` | Model-agnostic feature importance |